# Русский ModernBERT: дообучение выбора приложения

Проверяем компактный русский reranker на нашей задаче: запрос → E5 top-20 → выбранный ID. Модель уже знает русский; адаптируем выбор приложения, файла или папки.

Те же 22 656 обучающих строк (5 664 clean, 16 992 STT) и 150 синтетических validation. Связанные группы не пересекаются. Test и личные каталоги не входят в этот пакет.

Короткий чистый старт: первые 2 272 строки, затем смесь оставшихся; максимум один проход. Выбор checkpoint по validation NLL каждые 142 обновления; patience=3, min_delta=0.005. Исходные веса также участвуют в выборе. Это один исследовательский запуск, не поиск гиперпараметров.

## Среда

Закреплённые веса и зависимости; родная скалярная голова, без генерации текста. FP32. Скорость на GPU не подтверждает скорость CPU.

In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.12.1', 'safetensors==0.6.2'])
import torch
assert torch.cuda.is_available(), 'Нужен GPU Kaggle'
print({'torch':torch.__version__, 'gpu':torch.cuda.get_device_name(0)})


## Закреплённые данные и обучение

На входе пары запроса и карточки. Двадцать оценок сравниваются cross-entropy; target ID используется только для вычисления loss. Порядок кандидатов не исправляется. Промахи E5 остаются ошибками цепочки и не превращаются в ложные метки для stopping.

In [ ]:
matches = list(Path('/kaggle/input').rglob('modernbert-input-manifest.json'))
assert len(matches)==1, f'Ожидался один пакет данных, найдено {len(matches)}'
input_dir = matches[0].parent
assert hashlib.sha256(matches[0].read_bytes()).hexdigest()=='cabb5d2913fb4ac25ed70ee9e9bdb6d4a0c592cf6fef5a8464be5431f0eb5782', 'Другой пакет данных'
output_dir = Path('/kaggle/working/results')
command = [sys.executable, str(input_dir/'train_modernbert_kaggle.py'), '--input', str(input_dir), '--output', str(output_dir)]
with Path('/kaggle/working/training.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    assert process.wait()==0, 'Ошибка обучения; см. training.log'


## Результаты

Сравниваем исходную и выбранную модель по каждому профилю. Если лучший шаг равен нулю, улучшения дообучением не подтверждены. После фиксации весов измеряем остальные наборы и CPU latency.

In [ ]:
result = json.loads((output_dir/'result.json').read_text())
print({k:result[k] for k in ('steps','best_step','stop_reason','elapsed_seconds')})
for profile in result['baseline']['metrics']:
    before = result['baseline']['metrics'][profile]['accuracy']
    after = result['best']['metrics'][profile]['accuracy']
    print(f'{profile}: {before:.1%} → {after:.1%}')
